In [ ]:
# LOC Isolation Forest + SHAP — ICM Split (icm_ever_owned = 1 vs 0)
# Source: tmp_7d.hce_adr_avtar_like_25_26_f
# Runs the production pipeline separately for each ICM cohort

from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import shap
import matplotlib.pyplot as plt

In [ ]:
engine = create_engine(URL(
    account       = "UHG-UHGDWAAS",
    user          = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role          = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse     = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database      = "VING_PRD_TREND_DB",
    schema        = "TMP_1M"
))

In [ ]:
# ── Data Pull — both ICM cohorts

def pull_data(engine, icm_value, year_range, year_label):
    """Pull hospital_group × market aggregation for a given ICM cohort and year."""
    member_appeal_col = (
        ", count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt"
        if year_label == '2026' else ""
    )
    sql = f"""
        select
            d.collection as hospital_group
            , a.fin_market
            , '{year_label}' as year
            , count(distinct a.case_id) as case_count
            , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
            , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
            , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
            , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
            {member_appeal_col}
        from tmp_7d.hce_adr_avtar_like_25_26_f as a
        left join tmp_1y.tin_collection as d
            on substr(a.fa_prov_id, 2, 9) = d.tin
        where a.fin_brand = 'M&R'
            and a.icm_ever_owned = {icm_value}
            and a.svc_setting = 'Inpatient'
            and a.admit_cat_cd in ('17 - Medical', '30 - Surgical', '33 - Transplant')
            and to_varchar(a.admit_dt_act, 'MM/dd/yyyy') is not null
            and (
                (a.global_cap = 'NA' and a.sgr_source_name = 'COSMOS'
                 and a.fin_product_level_3 != 'INSTITUTIONAL' and a.tfm_include_flag = 1)
                or (a.sgr_source_name = 'NICE' and a.nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
            )
            and d.collection is not null
            and a.admit_act_month between '{year_range[0]}' and '{year_range[1]}'
        group by 1, 2
        having count(distinct a.case_id) >= 30
    """
    return pd.read_sql(sql, engine)

# Pull for both cohorts
data = {}
for icm_label, icm_val in [('ICM-owned', 1), ('Non-ICM', 0)]:
    df_25 = pull_data(engine, icm_val, ('202501', '202512'), '2025')
    df_26 = pull_data(engine, icm_val, ('202601', '202604'), '2026')
    data[icm_label] = {'2025': df_25, '2026': df_26}
    print(f"{icm_label}: 2025 = {len(df_25)} rows | 2026 = {len(df_26)} rows")

In [ ]:
# ── Pipeline Functions

# Rate configs (reduced: no appeal_overturn, no mcr_overturn)
rate_configs = {
    'initial_adr':    {'num': 'initial_adr_cnt',    'denom': 'case_count'},
    'persistent_adr': {'num': 'persistent_adr_cnt', 'denom': 'case_count'},
    'p2p_overturn':   {'num': 'p2p_ovrtn_cnt',     'denom': 'initial_adr_cnt'},
    'md_review':      {'num': 'md_reviewed_cnt',    'denom': 'case_count'},
}

iso_kpi_6 = [
    'log_case_count', 'log_initial_adr_cnt',
    'initial_adr_adj_rate', 'persistent_adr_adj_rate',
    'p2p_overturn_adj_rate', 'md_review_adj_rate'
]
iso_kpi_7 = iso_kpi_6 + ['member_appeal_adj_rate']


def estimate_k_mom(rates, denoms):
    mask = np.isfinite(rates) & (denoms > 0)
    r, n = rates[mask], denoms[mask]
    if len(r) < 10:
        return 50.0
    p_bar = r.mean()
    v_obs = r.var(ddof = 1)
    n_harm = len(n) / np.sum(1.0 / n)
    v_samp = p_bar * (1 - p_bar) / n_harm
    v_signal = v_obs - v_samp
    if v_signal <= 0:
        return 50.0
    alpha = p_bar * (p_bar * (1 - p_bar) / v_signal - 1)
    beta = (1 - p_bar) * (p_bar * (1 - p_bar) / v_signal - 1)
    if alpha <= 0 or beta <= 0:
        return 50.0
    return alpha + beta


def run_pipeline(df_25, df_26, label):
    """Run full IF pipeline for one ICM cohort. Returns dict of results."""
    df = pd.concat([df_25, df_26], ignore_index = True)

    # k estimation (pooled)
    k_map = {}
    for feat, cfg in rate_configs.items():
        rates = df[cfg['num']] / df[cfg['denom']].replace(0, np.nan)
        k_map[feat] = estimate_k_mom(rates.values, df[cfg['denom']].values)

    # member appeal k (2026 only)
    df_26_only = df[df['year'] == '2026'].copy()
    if 'member_appeal_cnt' in df_26_only.columns:
        rates_ma = df_26_only['member_appeal_cnt'] / df_26_only['initial_adr_cnt'].replace(0, np.nan)
        k_map['member_appeal'] = estimate_k_mom(rates_ma.values, df_26_only['initial_adr_cnt'].values)

    # global rates (full >=30 pop)
    global_rates = {}
    for feat, cfg in rate_configs.items():
        global_rates[feat] = df[cfg['num']].sum() / df[cfg['denom']].sum()
    if 'member_appeal_cnt' in df_26_only.columns:
        global_rates['member_appeal'] = df_26_only['member_appeal_cnt'].sum() / df_26_only['initial_adr_cnt'].sum()

    # shrinkage (>=10 ADR filter)
    def shrink(num, denom, k, p_global):
        return (num + k * p_global) / (denom + k)

    df_model = df[df['initial_adr_cnt'] >= 10].copy()
    for feat, cfg in rate_configs.items():
        df_model[f"{feat}_adj_rate"] = shrink(df_model[cfg['num']], df_model[cfg['denom']], k_map[feat], global_rates[feat])

    if 'member_appeal_cnt' in df_model.columns:
        df_model['member_appeal_adj_rate'] = np.where(
            df_model['year'] == '2026',
            shrink(df_model['member_appeal_cnt'], df_model['initial_adr_cnt'], k_map['member_appeal'], global_rates['member_appeal']),
            np.nan
        )

    df_model = df_model.assign(
        log_case_count = lambda x: np.log1p(x['case_count']),
        log_initial_adr_cnt = lambda x: np.log1p(x['initial_adr_cnt'])
    )

    # split by year
    df_25m = df_model[df_model['year'] == '2025'].copy()
    df_26m = df_model[df_model['year'] == '2026'].copy()

    # bootstrap stability
    def bootstrap_flags(df_year, features, n_iter = 200, pct = 2.5, random_state = 42):
        rng = np.random.RandomState(random_state)
        n = len(df_year)
        if n < 10:
            return np.zeros(n)
        flag_counts = np.zeros(n)
        for i in range(n_iter):
            idx = rng.choice(n, size = n, replace = True)
            X_boot = df_year[features].iloc[idx].values
            model = IsolationForest(n_estimators = 300, contamination = 'auto', random_state = i)
            model.fit(X_boot)
            scores = model.decision_function(df_year[features].values)
            threshold = np.percentile(scores, pct)
            flag_counts += (scores <= threshold).astype(int)
        return flag_counts / n_iter

    print(f"  {label} — bootstrapping 2025 ({len(df_25m)} hospitals)...")
    df_25m['bootstrap_flag_pct'] = bootstrap_flags(df_25m, iso_kpi_6)
    print(f"  {label} — bootstrapping 2026 ({len(df_26m)} hospitals)...")
    df_26m['bootstrap_flag_pct'] = bootstrap_flags(df_26m, iso_kpi_7)

    df_25m['consensus_flag'] = df_25m['bootstrap_flag_pct'] >= 0.50
    df_26m['consensus_flag'] = df_26m['bootstrap_flag_pct'] >= 0.50

    # SHAP
    def compute_shap(df_year, features):
        model = IsolationForest(n_estimators = 300, contamination = 'auto', random_state = 42)
        model.fit(df_year[features])
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(df_year[features])
        min_idx = np.argmin(shap_vals, axis = 1)
        drivers = [features[i] for i in min_idx]
        medians = {f: df_year[f].median() for f in features}
        directions = [
            'HIGH' if df_year[feat].iloc[row] > medians[feat] else 'LOW'
            for row, feat in enumerate(drivers)
        ]
        df_year['shap_driver'] = drivers
        df_year['driver_direction'] = directions
        return df_year, shap_vals

    df_25m, shap_25 = compute_shap(df_25m, iso_kpi_6)
    df_26m, shap_26 = compute_shap(df_26m, iso_kpi_7)

    print(f"  {label} — 2025 flagged: {df_25m['consensus_flag'].sum()} | 2026 flagged: {df_26m['consensus_flag'].sum()}")

    return {
        '2025': df_25m, '2026': df_26m,
        'shap_25': shap_25, 'shap_26': shap_26,
        'k_map': k_map
    }

print("Pipeline functions defined.")

In [ ]:
# ── Run Pipeline for Both Cohorts

results = {}
for icm_label, icm_val in [('ICM-owned', 1), ('Non-ICM', 0)]:
    print(f"\n{'=' * 60}")
    print(f"Running pipeline: {icm_label} (icm_ever_owned = {icm_val})")
    print(f"{'=' * 60}")
    results[icm_label] = run_pipeline(
        data[icm_label]['2025'],
        data[icm_label]['2026'],
        icm_label
    )

In [ ]:
# ── Results Comparison

DRIVER_RENAME = {
    'md_review_adj_rate': 'MD Escalation',
    'member_appeal_adj_rate': 'Member Appeal',
    'persistent_adr_adj_rate': 'Persistent ADR',
    'initial_adr_adj_rate': 'Initial ADR',
    'p2p_overturn_adj_rate': 'P2P Overturn',
    'log_case_count': 'Case Count',
    'log_initial_adr_cnt': 'Initial ADR Count',
}

for icm_label in ['ICM-owned', 'Non-ICM']:
    for yr in ['2025', '2026']:
        df_yr = results[icm_label][yr]
        flagged = df_yr[df_yr['consensus_flag']].sort_values('bootstrap_flag_pct', ascending = False).copy()
        if len(flagged) == 0:
            print(f"\n{icm_label} | {yr}: No hospitals flagged")
            continue
        flagged['driver_label'] = flagged['shap_driver'].map(DRIVER_RENAME).fillna(flagged['shap_driver'])
        out = flagged[['hospital_group', 'fin_market', 'case_count', 'bootstrap_flag_pct', 'driver_label', 'driver_direction']].copy()
        out.columns = ['Hospital Group', 'Market', 'Cases', 'Boot %', 'Driver', 'Dir']
        out['Boot %'] = (out['Boot %'] * 100).round(1)

        print(f"\n{'=' * 80}")
        print(f" {icm_label} | {yr} — FLAGGED ({len(flagged)})")
        print(f"{'=' * 80}")
        print(out.to_string(index = False))

In [ ]:
# ── Overlap Analysis: hospitals flagged in both cohorts

for yr in ['2025', '2026']:
    icm_set = set(
        results['ICM-owned'][yr].query('consensus_flag')[['hospital_group', 'fin_market']].apply(tuple, axis = 1)
    )
    non_icm_set = set(
        results['Non-ICM'][yr].query('consensus_flag')[['hospital_group', 'fin_market']].apply(tuple, axis = 1)
    )
    overlap = icm_set & non_icm_set
    icm_only = icm_set - non_icm_set
    non_icm_only = non_icm_set - icm_set

    print(f"\n{yr} Overlap:")
    print(f"  ICM-owned flagged: {len(icm_set)}")
    print(f"  Non-ICM flagged:   {len(non_icm_set)}")
    print(f"  Both:              {len(overlap)}")
    print(f"  ICM-only:          {len(icm_only)}")
    print(f"  Non-ICM-only:      {len(non_icm_only)}")
    if overlap:
        print(f"  Shared hospitals:")
        for h, m in sorted(overlap):
            print(f"    {h} | {m}")

In [ ]:
# ── SHAP Summary Plots (side-by-side: ICM vs Non-ICM)
fig, axes = plt.subplots(2, 2, figsize = (14, 10))

for col, icm_label in enumerate(['ICM-owned', 'Non-ICM']):
    for row, (yr, feats) in enumerate([('2025', iso_kpi_6), ('2026', iso_kpi_7)]):
        ax = axes[row, col]
        plt.sca(ax)
        shap_key = f'shap_{yr[-2:]}'
        df_yr = results[icm_label][yr]
        shap.summary_plot(
            results[icm_label][shap_key], df_yr[feats],
            feature_names = feats, plot_type = 'bar', show = False
        )
        ax.set_title(f"{icm_label} — {yr}")

plt.tight_layout()
plt.show()